# Is this video AI-generated? · É IA?

Run the [AI Video Observatory](https://github.com/lenaschwantes/ai-video-observatory) analyzer here, with nothing to install.
Rode o analisador aqui, sem instalar nada.

1. **Copy & Edit** this notebook (top right). / Clique em **Copy & Edit**.
2. **Settings → Internet: on**, and add a free key under **Add-ons → Secrets**: `GROQ_API_KEY` ([get one](https://console.groq.com/keys)) or `GEMINI_API_KEY` ([get one](https://aistudio.google.com/apikey)).
   Ative a internet e adicione uma chave gratuita em **Add-ons → Secrets**.
3. Paste the video link in the third cell and **Run all**. / Cole o link e rode tudo.

Your key stays in Kaggle Secrets; this notebook never prints it. The video is deleted after the analysis.
A chave fica nos Secrets do Kaggle e nunca é exibida. O vídeo é apagado depois da análise.


In [ ]:
# 1. Install the analyzer (about a minute)
!pip install -q "ai-video-observatory[detect] @ git+https://github.com/lenaschwantes/ai-video-observatory"


In [ ]:
# 2. Load your key from Kaggle Secrets (never printed)
import os

try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    for _name in ("GROQ_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass  # that secret is not set
except ImportError:
    pass  # not on Kaggle: keys come from your environment

from observatory.detect.vision import list_vision_models

# Groq and Claude are picked automatically; Gemini and OpenAI need a model name,
# so ask the key which vision models it can use and take the first.
if not (os.environ.get("GROQ_API_KEY") or os.environ.get("ANTHROPIC_API_KEY")):
    for _prov, _key, _var in (("gemini", "GEMINI_API_KEY", "GEMINI_VISION_MODEL"), ("openai", "OPENAI_API_KEY", "OPENAI_VISION_MODEL")):
        if os.environ.get(_key):
            os.environ["VISION_PROVIDER"] = _prov
            os.environ.setdefault(_var, list_vision_models(_prov, os.environ[_key])[0])
            break

from observatory.detect.vision import model_name, provider

print("Visual check:", f"{provider()} / {model_name()}" if provider() else "OFF (add a key in Add-ons → Secrets)")


In [ ]:
# 3. Paste the video link (YouTube, TikTok, Instagram or X) / Cole o link do vídeo
VIDEO_URL = "https://www.youtube.com/watch?v=WeIzH2jFsNw"

# Platforms sometimes block downloads from cloud machines. If that happens,
# upload the video with "Add data" and put its path here instead:
VIDEO_FILE = ""  # e.g. "/kaggle/input/my-video/clip.mp4"


In [ ]:
# 4. Analyze
from pathlib import Path

from observatory.detect.pipeline import analyze, analyze_file

REPORTS = Path("/kaggle/working/reports") if Path("/kaggle/working").exists() else Path("reports")
report = analyze_file(Path(VIDEO_FILE), REPORTS) if VIDEO_FILE else analyze(VIDEO_URL, REPORTS)
print(f"{report.probability:.0%} -> {report.verdict.value}")


In [ ]:
# 5. The report, with the evidence frames / O relatório, com os frames
import html

from IPython.display import HTML, display

LABEL = {"likely_ai": "Likely AI · Provavelmente IA", "inconclusive": "Inconclusive · Inconclusivo",
         "likely_human": "Likely real · Provavelmente real"}
out = [f"<h2>{html.escape(report.video.title or report.slug)}</h2>",
       f"<p style='font-size:2.4rem;font-weight:700;margin:0'>{report.probability:.0%}</p>",
       f"<p><b>{LABEL[report.verdict.value]}</b> · weights {report.method_version}</p>"]
for s in report.signals:
    state = s.direction.value if s.available else "n/a"
    out.append(f"<h4>{html.escape(s.label)} <small>({state})</small></h4>")
    if s.note:
        out.append(f"<p style='color:#666'>{html.escape(s.note)}</p>")
    for e in s.evidence:
        when = f"{e.timestamp:.1f}s · " if e.timestamp is not None else ""
        out.append(f"<p>• {when}<b>{e.direction.value}</b>: {html.escape(e.text)}</p>")
        if e.frame:
            src = (REPORTS / report.slug / e.frame).as_posix()
            out.append(f"<img src='{src}' style='max-width:360px;border-radius:8px'>")
out.append(f"<p style='color:#666'>{html.escape(report.disclaimer)}</p>")
display(HTML("".join(out)))
print("JSON:", REPORTS / report.slug / "report.json")
